# CS-781 Phase 5 - results table and error analysis (CPU is enough, a few minutes)

Uses the cached dev scores of the best full-data runs (nothing is re-run). Part A: every cached run of every notebook in one table. Part B, on the selection set: overview, threshold strategies (global / per-statute / shrunk), precision-recall trade-off, calibration, per-statute F1 against training support, F1 by number of gold statutes and by case length, label relationships (co-predicted chains, substitutions, missed statutes), where the document head looks, and model agreement. Plots go to the working directory and the bucket.

In [1]:
TOKENIZER_NAME = None
!pip install -q -U "huggingface_hub>=1.5"
import os, json, shutil, subprocess, sys, time, zipfile, random, math, gc
from collections import Counter, defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download, login, download_bucket_files, batch_bucket_files
from kaggle_secrets import UserSecretsClient

NB_START = time.time()
WORK = "/kaggle/working"
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_BUCKET = "kavincg23/cs781-models"
HF_CHECKPOINT_REPO = "kavincg23/cs781-checkpoints"   # old repo, read-only fallback
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
HF_WRITE_TOKEN = user_secrets.get_secret("HF_WRITE")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def get_dataset_file(filename):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename, repo_type="dataset", token=HF_TOKEN)


def fetch_model_file(name, local_path=None):
    local_path = local_path or f"{WORK}/{name}"
    if os.path.exists(local_path):
        return local_path
    part = local_path + ".part"
    try:
        download_bucket_files(HF_BUCKET, files=[(name, part)], raise_on_missing_files=True, token=HF_WRITE_TOKEN)
        os.replace(part, local_path)
        return local_path
    except Exception:
        if os.path.exists(part):
            os.remove(part)
    # fallback: the OLD model repo (the Phase-1 notebooks do the same; the tokenizer only lives there)
    p = hf_hub_download(repo_id=HF_CHECKPOINT_REPO, filename=name, repo_type="model", token=HF_WRITE_TOKEN)
    shutil.copy(p, local_path)
    print(f"(found {name} in the old repo {HF_CHECKPOINT_REPO})")
    return local_path


def upload_model_file(local_path, name=None):
    name = name or os.path.basename(local_path)
    try:
        batch_bucket_files(HF_BUCKET, add=[(local_path, name)], token=HF_WRITE_TOKEN)
        print(f"uploaded {name} to bucket {HF_BUCKET}")
    except Exception as e:
        print(f"(upload of {name} failed: {type(e).__name__}: {e}; the file is still in {WORK})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 11.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.3.0 which is incompatible.
transformers 5.16.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 which is incompatible.
tokenizers 0.23.1 requires huggingface-hub<2.0,>=0.16.4, but you have huggingface-hub 2.3.0 which is incompatible.


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


device: cpu 


In [2]:
# ===================== Phase 5 (Legal Statute Identification) engine: long-document chunking, chunk pooling, multi-label document heads, losses, thresholds, training, inference =====================
# Works with any Phase-1 model wrapped as a "backbone" exposing  features(ids, mask, soft_prompt) -> (B, T, d_model).
# A document = its BPE token ids. It is cut into CHUNKS of <= WINDOW tokens (sliding window, stride STRIDE); at most MAX_CHUNKS chunks are kept
# (SELECT says which). The backbone reads every chunk as one row, a chunk is pooled to one vector (mean of its tokens and/or its last token), a document
# head (mean / attention / BiLSTM + attention / one attention per statute) turns the chunk vectors into L statute scores (logits, one per statute id).
import math, time, random, json, os, copy, zipfile, hashlib
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint


# ----------------------------------------------------------------------------------------------------------------------
# 1. Documents, chunks
# ----------------------------------------------------------------------------------------------------------------------
def doc_text(r):
    t = r["text"]
    return " ".join(t) if isinstance(t, list) else str(t)


def text_hash(t):
    return hashlib.md5((t[:2000] + "||" + t[-2000:]).encode("utf-8", errors="ignore")).hexdigest()


def label_list(v):
    return sorted(set(int(x) for x in (v if isinstance(v, (list, tuple)) else [v])))


def multi_hot(labels, L):
    y = np.zeros(L, dtype=np.uint8)
    if len(labels):
        y[list(labels)] = 1
    return y


def encode_rows(tok, rows, id_key, label_key, max_tokens, log=print):
    """-> list of dict(id, ids=np.uint16 array, labs=list|None, n). Documents longer than max_tokens keep their first and last max_tokens//2 tokens."""
    out, t0, cut = [], time.time(), 0
    for i, r in enumerate(rows):
        ids = tok.encode(doc_text(r))
        if not ids:
            ids = [1]
        if len(ids) > max_tokens:
            ids = ids[:max_tokens // 2] + ids[-(max_tokens - max_tokens // 2):]; cut += 1
        out.append(dict(id=r[id_key], ids=np.asarray(ids, dtype=np.uint16), labs=label_list(r[label_key]) if label_key and r.get(label_key) is not None else None, n=len(ids)))
        if (i + 1) % 5000 == 0:
            log(f"    tokenised {i + 1}/{len(rows)} documents [{time.time() - t0:.0f}s]")
    return out, dict(documents=len(out), tokens=int(sum(d["n"] for d in out)), cut=cut)


def cue_positions(ids, patterns):
    """Start positions of the cue token patterns (the BPE tokens of <SECTION> / <ACT>: where the masked statute mentions were) in a token array."""
    a = np.asarray(ids); out = []
    for p in patterns:
        k = len(p)
        if k == 0 or len(a) < k:
            continue
        m = np.ones(len(a) - k + 1, dtype=bool)
        for j, t in enumerate(p):
            m &= a[j:len(a) - k + 1 + j] == t
        out.append(np.flatnonzero(m))
    return np.unique(np.concatenate(out)) if out else np.zeros(0, dtype=np.int64)


def make_spans(n, C, d=None):
    """Chunk spans (start, end) of a document with n tokens, after applying MAX_CHUNKS / SELECT.
    SELECT='cue' (evidence-based selection): keep the MAX_CHUNKS chunks with the most cue tokens (<SECTION> / <ACT> mentions); ties -> head+tail order."""
    W, S = C["WINDOW"], max(1, C["STRIDE"])
    if n <= W:
        return [(0, n)]
    starts = list(range(0, n - W + 1, S))
    if starts[-1] + W < n:
        starts.append(n - W)                                   # the last chunk is aligned to the end of the document
    spans = [(s, s + W) for s in starts]
    K, m, how = len(spans), C["MAX_CHUNKS"], C["SELECT"]
    if how == "all" or m is None or K <= m:
        return spans
    if how == "first":
        idx = list(range(m))
    elif how == "last":
        idx = list(range(K - m, K))
    elif how == "middle":
        a = (K - m) // 2; idx = list(range(a, a + m))
    elif how == "head_tail":
        h = min(C["HEAD_CHUNKS"], m - 1) if m > 1 else 0
        idx = list(range(h)) + list(range(K - (m - h), K))
    elif how == "uniform":
        idx = sorted(set(int(round(x)) for x in np.linspace(0, K - 1, m)))
    elif how == "cue":
        if d is not None and "cue" not in d:
            d["cue"] = cue_positions(d["ids"], C["CUE_PATTERNS"])
        cue = d["cue"] if d is not None else np.zeros(0)
        h = min(C["HEAD_CHUNKS"], m - 1) if m > 1 else 0
        ht = set(list(range(h)) + list(range(K - (m - h), K)))
        score = [int(((cue >= s) & (cue < e)).sum()) for s, e in spans]
        idx = sorted(sorted(range(K), key=lambda i: (-score[i], 0 if i in ht else 1, i))[:m])
    else:
        raise ValueError(how)
    return [spans[i] for i in idx]


def collate(docs, idxs, C, with_labels=True, pad_id=0):
    rows, counts, spans_all = [], [], []
    for di in idxs:
        d = docs[di]; sp = make_spans(d["n"], C, d)
        for s, e in sp:
            rows.append(d["ids"][s:e].tolist())
        counts.append(len(sp)); spans_all.append(sp)
    R, T = len(rows), max(len(r) for r in rows)
    ids = torch.full((R, T), pad_id, dtype=torch.long)
    mask = torch.zeros((R, T), dtype=torch.bool)
    for r, a in enumerate(rows):
        ids[r, :len(a)] = torch.tensor(a); mask[r, :len(a)] = True
    K = max(counts)
    chunk_mask = torch.arange(K).unsqueeze(0) < torch.tensor(counts).unsqueeze(1)
    b = dict(ids=ids, mask=mask, counts=counts, chunk_mask=chunk_mask, spans=spans_all)
    if with_labels and all(docs[di].get("y") is not None for di in idxs):
        b["labels"] = torch.from_numpy(np.stack([docs[di]["y"] for di in idxs])).float()
    if with_labels and all(docs[di].get("kd") is not None for di in idxs):
        b["kd"] = torch.stack([torch.as_tensor(docs[di]["kd"]).float() for di in idxs])
    return b


def to_device(b, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in b.items()}


# ----------------------------------------------------------------------------------------------------------------------
# 2. Model = backbone + chunk pooling + multi-label document head
# ----------------------------------------------------------------------------------------------------------------------
class DocHead(nn.Module):
    """chunk vectors (B, K, d_in) + chunk mask -> statute logits (B, L).
    kind: 'mean' | 'attn' (one attention pooling shared by all statutes) | 'bilstm' (BiLSTM over the chunk sequence, then shared attention pooling)
          | 'label_attn' (every statute has its own attention query over the chunks: it can look at the chunk that is about it).
    label_mix: a zero-initialised L x L layer on the sigmoid of the first-stage scores lets the head learn label co-occurrence (e.g. statute 12 -> 13)."""

    def __init__(self, d_in, n_labels, kind="bilstm", hidden=256, dropout=0.1, norm=True, label_mix=False):
        super().__init__()
        self.kind, self.L = kind, n_labels
        self.norm = nn.LayerNorm(d_in) if norm else nn.Identity()
        self.drop = nn.Dropout(dropout)
        d = d_in
        self.lstm = None
        if kind == "bilstm":
            self.lstm = nn.LSTM(d_in, hidden, batch_first=True, bidirectional=True)
            d = 2 * hidden
        if kind in ("attn", "bilstm"):
            self.score = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
            self.out = nn.Linear(d, n_labels)
        elif kind == "label_attn":
            self.key = nn.Sequential(nn.Linear(d, 128), nn.Tanh())
            self.lq = nn.Parameter(torch.randn(n_labels, 128) * 0.1)
            self.W = nn.Parameter(torch.randn(n_labels, d) * (1.0 / math.sqrt(d)))
            self.b = nn.Parameter(torch.zeros(n_labels))
        elif kind in ("mean", "chunk_max", "chunk_mean"):     # chunk_*: statute scores for EVERY chunk, then max / mean over the chunks
            self.out = nn.Linear(d, n_labels)
        else:
            raise ValueError(kind)
        self.mix = nn.Linear(n_labels, n_labels, bias=False) if label_mix else None
        if self.mix is not None:
            nn.init.zeros_(self.mix.weight)
        self.last_attn = None

    @torch.no_grad()
    def init_bias(self, prior):
        """Output bias = logit of the label frequency: training starts from 'every statute at its base rate', not from p = 0.5 for all 100 statutes."""
        b = torch.logit(torch.as_tensor(prior, dtype=torch.float32).clamp(1e-4, 1 - 1e-4))
        (self.b if self.kind == "label_attn" else self.out.bias).copy_(b)

    def forward(self, v, mask):
        x = self.drop(self.norm(v.float()))
        if self.lstm is not None:
            lengths = mask.sum(1).clamp(min=1).cpu()
            packed = nn.utils.rnn.pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
            with torch.autocast(device_type=x.device.type, enabled=False):
                o, _ = self.lstm(packed)
            x, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=v.shape[1])
        if self.kind in ("chunk_max", "chunk_mean"):
            cl = self.out(x)                                                        # (B, K, L) chunk-level statute scores
            if self.kind == "chunk_max":
                cl = cl.masked_fill(~mask.unsqueeze(-1), float("-inf")); logits, am = cl.max(1)
                self.last_attn = torch.stack([(am == k).float().mean(-1) for k in range(cl.shape[1])], 1).detach()   # share of statutes whose max is in chunk k
            else:
                w = mask.float() / mask.float().sum(1, keepdim=True).clamp(min=1); self.last_attn = w.detach()
                logits = (cl * w.unsqueeze(-1)).sum(1)
        elif self.kind == "label_attn":
            s = torch.einsum("bkh,lh->bkl", self.key(x), self.lq).masked_fill(~mask.unsqueeze(-1), float("-inf"))
            w = torch.softmax(s, dim=1)                                            # (B, K, L): each statute's attention over the chunks
            self.last_attn = w.mean(-1).detach()
            logits = (w * torch.einsum("bkd,ld->bkl", x, self.W)).sum(1) + self.b
        else:
            if self.kind == "mean":
                w = mask.float() / mask.float().sum(1, keepdim=True).clamp(min=1)
            else:
                s = self.score(x).squeeze(-1).masked_fill(~mask, float("-inf"))
                w = torch.softmax(s, dim=1)
            self.last_attn = w.detach()
            logits = self.out((x * w.unsqueeze(-1)).sum(1))
        if self.mix is not None:
            logits = logits + self.mix(torch.sigmoid(logits))
        return logits


class DocModel(nn.Module):
    """Attribute names shared with the Phase-3/4 model (backbone, soft_prompt, head_parameters) so the technique code works unchanged."""

    def __init__(self, backbone, d_model, C):
        super().__init__()
        self.backbone = backbone
        self.soft_prompt = None
        self.pool = C["POOL"]
        self.row_group = C["ROW_GROUP"]
        self.ckpt = C.get("CKPT", False)
        if self.ckpt and any(getattr(m, "bias_rate", 0) > 0 or (hasattr(m, "tau") and getattr(type(m)._route, "__name__", "") != "_mor_route_frozen") for m in backbone.modules()):
            print("note: the backbone updates routing state inside its forward pass (MoE bias / MoR threshold): activation checkpointing switched off")
            self.ckpt = False
        d_pool = d_model * (2 if self.pool == "mean+last" else 1)
        self.head = DocHead(d_pool, C["N_LABELS"], C["DOC_HEAD"], C["HEAD_LSTM_HIDDEN"], C["HEAD_DROPOUT"], C["HEAD_NORM"], C.get("LABEL_MIX", False))

    def head_parameters(self):
        return list(self.head.parameters())

    def _pool_rows(self, ids, mask):
        h = self.backbone.features(ids, mask, self.soft_prompt).float()
        parts = []
        if self.pool in ("mean", "mean+last"):
            m = mask.unsqueeze(-1).float()
            parts.append((h * m).sum(1) / m.sum(1).clamp(min=1))
        if self.pool in ("last", "mean+last"):
            last = mask.sum(1) - 1
            parts.append(h[torch.arange(h.shape[0], device=h.device), last])
        return torch.cat(parts, -1)

    def chunk_vectors(self, b):
        ids, mask = b["ids"], b["mask"]
        trainable = self.training and any(p.requires_grad for p in self.backbone.parameters()) or (self.training and self.soft_prompt is not None)
        out = []
        for a in range(0, ids.shape[0], self.row_group):
            i, m = ids[a:a + self.row_group], mask[a:a + self.row_group]
            L = int(m.sum(1).max()); i, m = i[:, :L], m[:, :L]
            if not trainable:
                with torch.no_grad():
                    out.append(self._pool_rows(i, m))
            elif self.ckpt:
                out.append(checkpoint(self._pool_rows, i, m, use_reentrant=False))
            else:
                out.append(self._pool_rows(i, m))
        return torch.cat(out, 0)                                              # (R, d_pool), documents in order

    def forward(self, b):
        v = self.chunk_vectors(b)
        cm = b["chunk_mask"]
        vp = v.new_zeros(cm.shape[0], cm.shape[1], v.shape[1])
        vp[cm] = v                                                            # row-major order of True entries == document order
        return self.head(vp, cm)                                              # (B, L)


# ----------------------------------------------------------------------------------------------------------------------
# 3. Scores and thresholds (macro-F1 over the L statutes is the primary metric; micro-F1 and subset accuracy are secondary)
# ----------------------------------------------------------------------------------------------------------------------
def ml_scores(Y, pred):
    """Y, pred: (N, L) 0/1 arrays. Per-statute F1 is 0 when a statute has no true and no predicted positive (never the case here: every statute has dev support)."""
    Y, pred = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    tp = (Y & pred).sum(0).astype(np.float64); fp = (~Y & pred).sum(0); fn = (Y & ~pred).sum(0)
    p = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0); r = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    f = np.divide(2 * p * r, p + r, out=np.zeros_like(tp), where=(p + r) > 0)
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    mp, mr = TP / max(TP + FP, 1), TP / max(TP + FN, 1)
    inter = (Y & pred).sum(1).astype(np.float64); den = Y.sum(1) + pred.sum(1)
    sample_f = np.divide(2 * inter, den, out=np.zeros_like(inter), where=den > 0)
    return dict(macro_F1=float(f.mean()), micro_F1=float(2 * mp * mr / max(mp + mr, 1e-12)), subset_acc=float((Y == pred).all(1).mean()), macro_P=float(p.mean()), macro_R=float(r.mean()),
                micro_P=float(mp), micro_R=float(mr), samples_F1=float(sample_f.mean()), hamming=float((Y != pred).mean()), mean_pred_labels=float(pred.sum(1).mean()),
                mean_true_labels=float(Y.sum(1).mean()), per_label_F1=f.tolist(), n=int(len(Y)))


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(np.asarray(x, dtype=np.float64), -30, 30)))


GRID = np.round(np.arange(0.05, 0.951, 0.01), 2)


def apply_thr(P, thr, min_one=True):
    """P (N, L) probabilities, thr scalar or (L,). Every case has at least one statute in the data: if none passes its threshold, the statute with the largest margin P - thr is predicted."""
    thr = np.broadcast_to(np.asarray(thr, dtype=np.float64), (P.shape[1],))
    pred = P > thr
    if min_one:
        none = ~pred.any(1)
        if none.any():
            am = (P[none] - thr).argmax(1); sub = np.zeros((none.sum(), P.shape[1]), dtype=bool); sub[np.arange(len(am)), am] = True; pred[none] = sub
    return pred


def f1_grid(Y, P, grid=GRID):
    """-> F (T, L): per-statute F1 at every global threshold of the grid."""
    Yb = np.asarray(Y).astype(bool); F_ = np.zeros((len(grid), Yb.shape[1]))
    ny = Yb.sum(0).astype(np.float64)
    for k, t in enumerate(grid):
        pr = P > t
        tp = (Yb & pr).sum(0); npred = pr.sum(0)
        F_[k] = np.divide(2.0 * tp, ny + npred, out=np.zeros(Yb.shape[1]), where=(ny + npred) > 0)
    return F_


def fit_thresholds(Y, P, mode="global", shrink=0.5, grid=GRID, smooth=5):
    """-> (L,) thresholds. global: one threshold for all statutes (the one maximising macro-F1); per_label: each statute its own best threshold (on a
    smoothed F1 curve, to avoid chasing noise); shrunk: per_label pulled towards the global threshold (shrink = weight of the per-label value)."""
    F_ = f1_grid(Y, P, grid); L = F_.shape[1]
    tg = float(grid[int(np.argmax(F_.mean(1)))])
    if mode == "global":
        return np.full(L, tg)
    k = max(1, smooth)
    Fs = np.apply_along_axis(lambda c: np.convolve(np.pad(c, (k // 2, k // 2), mode="edge"), np.ones(k) / k, mode="valid"), 0, F_) if k > 1 else F_
    tl = grid[np.argmax(Fs, 0)]
    return tl if mode == "per_label" else shrink * tl + (1 - shrink) * tg


THR_MODES = ("global", "per_label", "shrunk")


def cross_fit(Y, P, modes=THR_MODES, reps=6, min_one=True, seed=0):
    """Honest threshold evaluation: thresholds are fitted on one random half of the cases and scored on the other half. -> {mode: mean macro-F1}, {mode: sd}."""
    res = {m: [] for m in modes}
    for r in range(reps):
        perm = np.random.RandomState(seed + r).permutation(len(Y)); A, B = perm[:len(Y) // 2], perm[len(Y) // 2:]
        for m in modes:
            for tr, te in ((A, B), (B, A)):
                res[m].append(ml_scores(Y[te], apply_thr(P[te], fit_thresholds(Y[tr], P[tr], m), min_one))["macro_F1"])
    return {m: float(np.mean(v)) for m, v in res.items()}, {m: float(np.std(v)) for m, v in res.items()}


def best_global(Y, P, min_one=True):
    t = float(fit_thresholds(Y, P, "global")[0])
    return t, ml_scores(Y, apply_thr(P, t, min_one))["macro_F1"]


# ----------------------------------------------------------------------------------------------------------------------
# 4. Losses
# ----------------------------------------------------------------------------------------------------------------------
def pos_weights(Ytrain, power, cap):
    """pos_weight_l = ((N - n_l) / n_l) ** power, capped (power 0.5 = inverse square root of the odds)."""
    n = Ytrain.sum(0).astype(np.float64); N = len(Ytrain)
    w = np.minimum(((N - n) / np.maximum(n, 1)) ** power, cap)
    return torch.tensor(w / w.mean(), dtype=torch.float32)


def ml_loss(logits, y, C, pw=None):
    """LOSS: bce | bce_pw (pos_weight per statute) | focal | asl (asymmetric loss, Ridnik et al. 2021). Returns the mean over the batch of the per-statute mean x LOSS_SCALE."""
    eps = C.get("LABEL_SMOOTHING", 0.0)
    if eps:
        y = y * (1 - eps) + eps / 2
    kind = C["LOSS"]
    if kind in ("bce", "bce_pw"):
        l = F.binary_cross_entropy_with_logits(logits, y, pos_weight=pw if kind == "bce_pw" else None, reduction="none")
    elif kind == "focal":
        bce = F.binary_cross_entropy_with_logits(logits, y, reduction="none"); p = torch.sigmoid(logits)
        pt = y * p + (1 - y) * (1 - p)
        l = bce * (1 - pt) ** C["FOCAL_GAMMA"]
    elif kind == "asl":
        xp = torch.sigmoid(logits); xn = 1 - xp
        if C["ASL_CLIP"] > 0:
            xn = (xn + C["ASL_CLIP"]).clamp(max=1)
        lp = y * torch.log(xp.clamp(min=1e-8)); ln = (1 - y) * torch.log(xn.clamp(min=1e-8))
        pt = xp * y + xn * (1 - y)
        g = C["ASL_GAMMA_POS"] * y + C["ASL_GAMMA_NEG"] * (1 - y)
        l = -(lp + ln) * (1 - pt).detach().pow(g)
    else:
        raise ValueError(kind)
    return l.mean(1).mean() * C["LOSS_SCALE"]


def kd_loss(logits, teacher_logits, T, C):
    """Per-statute distillation: the student's tempered sigmoid is pulled to the teacher's tempered sigmoid (binary cross-entropy with soft targets, x T^2)."""
    soft = torch.sigmoid(teacher_logits / T)
    return F.binary_cross_entropy_with_logits(logits / T, soft, reduction="none").mean(1).mean() * T * T * C["LOSS_SCALE"]


# ----------------------------------------------------------------------------------------------------------------------
# 5. Training
# ----------------------------------------------------------------------------------------------------------------------
def _lr_lambda(total, warmup):
    def f(step):
        if step < warmup:
            return (step + 1) / max(1, warmup)
        return max(0.0, (total - step) / max(1, total - warmup))
    return f


def trainable_state(model):
    st = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters() if p.requires_grad}
    for n, bf in model.named_buffers():
        if n.endswith(("select_bias", ".tau", ".tau_ready", ".ada_mask")):
            st[n] = bf.detach().to("cpu", copy=True)
    return st


def load_trainable_state(model, st):
    known = dict(model.named_parameters()); known.update(dict(model.named_buffers()))
    missing = [k for k in st if k not in known]
    assert not missing, f"state has unknown keys: {missing[:5]}"
    model.load_state_dict(st, strict=False)


@torch.no_grad()
def predict_logits(model, docs, C, device, collect_attn=False):
    """-> (N, L) float32 logits in the order of docs, plus timing; with collect_attn also the per-document chunk attention weights and spans."""
    model.eval(); model.to(device)
    order = sorted(range(len(docs)), key=lambda i: len(make_spans(docs[i]["n"], C, docs[i])))        # similar chunk counts together: less padding
    out = np.zeros((len(docs), C["N_LABELS"]), dtype=np.float32)
    attn = {} if collect_attn else None
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    t0, tokens = time.time(), 0
    for a in range(0, len(order), C["INFER_BATCH"]):
        idx = order[a:a + C["INFER_BATCH"]]
        b = to_device(collate(docs, idx, C, with_labels=False), device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            lg = model(b)
        out[idx] = lg.float().cpu().numpy(); tokens += int(b["mask"].sum())
        if collect_attn:
            w = model.head.last_attn.cpu().numpy()
            for j, di in enumerate(idx):
                attn[docs[di]["id"]] = dict(w=w[j, :b["counts"][j]].tolist(), spans=b["spans"][j], n=docs[di]["n"])
    if device.type == "cuda":
        torch.cuda.synchronize()
    secs = time.time() - t0
    info = dict(seconds=round(secs, 2), tokens=tokens, tokens_per_sec=round(tokens / max(secs, 1e-9), 1), docs_per_sec=round(len(docs) / max(secs, 1e-9), 2))
    return (out, info, attn) if collect_attn else (out, info)


def train_doc(model, train_docs, dev_docs, C, device, log=print, dev_sub=None, ckpt_save=None, ckpt_load=None):
    """Trains on train_docs in EPOCHS 'epochs' of EPOCH_DOCS documents each (a stream over a reshuffled permutation, so a huge training set can be seen
    partially). After every epoch a fixed sample of the selection set is scored; the best epoch's state is kept (SELECT_METRIC: f1_tuned = macro-F1 at the
    best global threshold, f1 = at 0.5). Optional model hooks (AdaLoRA): extra_loss(), on_before_step(step, total)."""
    model.to(device)
    torch.manual_seed(C["SEED"]); random.seed(C["SEED"])
    N = len(train_docs)
    per_epoch = min(C["EPOCH_DOCS"] or N, N)
    epochs = C["EPOCHS"]
    Ytr = np.stack([d["y"] for d in train_docs])
    pw = pos_weights(Ytr, C["PW_POWER"], C["PW_CAP"]).to(device) if C["LOSS"] == "bce_pw" else None
    model.head.init_bias(Ytr.mean(0))
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    hp = set(id(p) for p in head_params)
    other = [p for p in model.parameters() if p.requires_grad and id(p) not in hp]
    groups = [dict(params=head_params, lr=C["HEAD_LR"])]
    if other:
        groups.append(dict(params=other, lr=C["LR"]))
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"], betas=(0.9, 0.98))
    steps_per_epoch = math.ceil(per_epoch / (C["BATCH"] * C["ACCUM"]))
    total = steps_per_epoch * epochs
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _lr_lambda(total, int(C["WARMUP_FRAC"] * total)))
    use_amp = device.type == "cuda" and C["AMP"] == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)
    kd_on = C.get("KD_ALPHA", 0) > 0 and all(d.get("kd") is not None for d in train_docs)
    rng = random.Random(C["SEED"] * 1000 + 7)
    stream = []
    if C.get("SAMPLING", "uniform") == "rare":      # rare-statute oversampling: a case is drawn with probability ~ max over its statutes of (1 / support) ** RARE_POWER
        sup = Ytr.sum(0).astype(np.float64); wl = (1.0 / np.maximum(sup, 1)) ** C.get("RARE_POWER", 0.5)
        wdoc = (Ytr * wl[None, :]).max(1); wdoc = (wdoc / wdoc.mean()).tolist()
    def take(k):
        nonlocal stream
        while len(stream) < k:
            if C.get("SAMPLING", "uniform") == "rare":
                stream += rng.choices(range(N), weights=wdoc, k=N)
            else:
                p = list(range(N)); rng.shuffle(p); stream += p
        out, stream = stream[:k], stream[k:]
        return out
    dev_idx = list(range(len(dev_docs)))
    if dev_sub and dev_sub < len(dev_docs):
        dev_idx = sorted(random.Random(1).sample(dev_idx, dev_sub))
    dev_eval = [dev_docs[i] for i in dev_idx]
    Yd = np.stack([d["y"] for d in dev_eval])
    hist, best, best_state, best_ep, bad = [], -1.0, None, 0, 0
    t0, tokens, gstep = time.time(), 0, 0
    start_ep = 1
    ck = ckpt_load() if ckpt_load else None
    if ck is not None:          # resume after the last finished epoch (a previous session was stopped)
        load_trainable_state(model, ck["model"])
        opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        rng.setstate(ck["rng"]); stream = ck["stream"]; torch.set_rng_state(ck["torch_rng"])
        if device.type == "cuda" and ck.get("cuda_rng") is not None:
            torch.cuda.set_rng_state(ck["cuda_rng"], device)
        hist, best, best_state, best_ep, bad = ck["hist"], ck["best"], ck["best_state"], ck["best_ep"], ck["bad"]
        tokens, gstep = ck["tokens"], ck["gstep"]; t0 = time.time() - ck["elapsed"]; start_ep = ck["epoch"] + 1
        log(f"  resumed from the checkpoint after epoch {ck['epoch']} (best so far: epoch {best_ep}, {best:.4f})")
    log(f"training: {N} train documents, {per_epoch} per epoch x {epochs} epochs, {steps_per_epoch} optimizer steps/epoch, loss {C['LOSS']}, head {C['DOC_HEAD']}" + (", KD on" if kd_on else "")
        + (f"; pos_weight range {float(pw.min()):.2f}-{float(pw.max()):.2f}" if pw is not None else ""))
    for ep in range(start_ep, epochs + 1):
        model.train()
        order = take(per_epoch)
        run_loss, nb, skipped, pending = 0.0, 0, 0, 0
        opt.zero_grad(set_to_none=True)
        for bi in range(0, len(order), C["BATCH"]):
            b = to_device(collate(train_docs, order[bi:bi + C["BATCH"]], C), device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(b)
            logits = logits.float()
            loss = ml_loss(logits, b["labels"], C, pw)
            if kd_on:
                loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd_loss(logits, b["kd"], C["KD_TEMPERATURE"], C)
            if hasattr(model, "extra_loss"):
                loss = loss + model.extra_loss()
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss / C["ACCUM"]).backward()
            nb += 1; pending += 1; tokens += int(b["mask"].sum()); run_loss += loss.item()
            if pending >= C["ACCUM"] or (bi + C["BATCH"] >= len(order) and pending > 0):
                scaler.unscale_(opt)
                if hasattr(model, "on_before_step"):
                    model.on_before_step(gstep, total)
                torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], C["GRAD_CLIP"])
                scaler.step(opt); scaler.update(); sched.step(); gstep += 1
                opt.zero_grad(set_to_none=True); pending = 0
        row = dict(epoch=ep, train_loss=round(run_loss / max(1, nb), 5), train_seconds=round(time.time() - t0, 1))
        if skipped:
            row["skipped_nonfinite_batches"] = skipped
        lg, einfo = predict_logits(model, dev_eval, C, device)
        P = sigmoid_np(lg)
        s05 = ml_scores(Yd, apply_thr(P, 0.5)); thr, ft = best_global(Yd, P)
        sc_tuned = ml_scores(Yd, apply_thr(P, thr))
        score = s05["macro_F1"] if C["SELECT_METRIC"] == "f1" else ft
        row.update(dev_macroF1=round(s05["macro_F1"], 4), dev_macroF1_tuned=round(ft, 4), tuned_thr=round(thr, 2), dev_microF1_tuned=round(sc_tuned["micro_F1"], 4),
                   dev_subset_acc_tuned=round(sc_tuned["subset_acc"], 4), eval_seconds=einfo["seconds"])
        if score > best:
            best, best_ep, best_state, bad = score, ep, trainable_state(model), 0
        else:
            bad += 1
        hist.append(row)
        log("  epoch %d: " % ep + ", ".join(f"{k}={v}" for k, v in row.items() if k != "epoch"))
        if ckpt_save and ep < epochs:
            ckpt_save(dict(epoch=ep, model=trainable_state(model), opt=opt.state_dict(), sched=sched.state_dict(), scaler=scaler.state_dict(),
                           rng=rng.getstate(), stream=list(stream), torch_rng=torch.get_rng_state(),
                           cuda_rng=torch.cuda.get_rng_state(device) if device.type == "cuda" else None,
                           hist=hist, best=best, best_state=best_state, best_ep=best_ep, bad=bad, tokens=tokens, gstep=gstep, elapsed=time.time() - t0))
        if nb == 0:
            log("  every batch of this epoch was non-finite: stopping"); break
        if C["PATIENCE"] and bad >= C["PATIENCE"]:
            log(f"  early stop: no improvement for {bad} epochs"); break
        if C.get("TIME_BUDGET_MIN") and (time.time() - t0) / 60 > C["TIME_BUDGET_MIN"]:
            log(f"  time budget of {C['TIME_BUDGET_MIN']} min reached: stopping after epoch {ep}"); break
    train_secs = time.time() - t0 - sum(r["eval_seconds"] for r in hist)
    peak = torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else None
    return dict(history=hist, best_epoch=best_ep, best_score=best, best_state=best_state, train_seconds=round(train_secs, 1),
                peak_mem_gb=round(peak, 3) if peak else None, tokens_per_sec=round(tokens / max(train_secs, 1e-9), 1))


# ----------------------------------------------------------------------------------------------------------------------
# 6. Submission
# ----------------------------------------------------------------------------------------------------------------------
def write_submission(ids, pred_matrix, out_dir, name, expected_ids, n_labels):
    """JSON list [{"id": ..., "pred": [statute ids]}] (one entry per test case, sorted unique integers in 0..L-1) + a zip with prediction.json at its root."""
    assert list(ids) == list(expected_ids) or set(ids) == set(expected_ids), "ids differ from test.json"
    assert len(set(ids)) == len(ids) == len(expected_ids), "duplicate or missing ids"
    pm = np.asarray(pred_matrix).astype(bool); assert pm.shape == (len(ids), n_labels)
    sub = [dict(id=i, pred=[int(x) for x in np.flatnonzero(row)]) for i, row in zip(ids, pm)]
    assert all(len(set(s["pred"])) == len(s["pred"]) and all(isinstance(x, int) and 0 <= x < n_labels for x in s["pred"]) for s in sub)
    os.makedirs(out_dir, exist_ok=True)
    jpath = os.path.join(out_dir, f"{name}_prediction.json")
    with open(jpath, "w", encoding="utf-8") as f:
        json.dump(sub, f)
    zpath = os.path.join(out_dir, f"submission_{name}.zip")
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jpath, arcname="prediction.json")
    return jpath, zpath


# ===================== The Phase-5 evaluator (from the course file evaluate_phase5.py; read_truth / read_pred / the three scores are unchanged, main() became a function) =====================
# Truth JSON: a list of objects, [{"id":"...", "gold":[0,2,5,...]}, ...]      Prediction JSON: [{"id":"...", "pred":[0,2,5,...]}, ...]
import argparse, json, sys
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}; max_label=-1
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or not isinstance(r['gold'],list): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        seen=set()
        for label in r['gold']:
            if isinstance(label,bool) or not isinstance(label,int) or label<0 or label in seen: raise ValueError(f'Invalid/duplicate label in truth item {n}.')
            seen.add(label); max_label=max(max_label,label)
        out[r['id']]=r['gold']
    return out,max_label+1

def read_pred(path,nlab):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or not isinstance(r['pred'],list) or r['id'] in out: raise ValueError(f'Invalid or duplicate prediction item {n}.')
        seen=set()
        for x in r['pred']:
            if isinstance(x,bool) or not isinstance(x,int) or x<0 or x>=nlab or x in seen: raise ValueError(f'Invalid/duplicate label in prediction item {n}.')
            seen.add(x)
        out[r['id']]=r['pred']
    return out

def official_eval(truth_path, prediction_path, verbose=True):
    """Same checks and the same three numbers as evaluate_phase5.py: Macro-F1, Micro-F1, Subset Accuracy."""
    t,n=read_truth(truth_path); p=read_pred(prediction_path,n)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=np.zeros((len(t),n),dtype=np.uint8); yp=np.zeros((len(t),n),dtype=np.uint8)
    for i,k in enumerate(t):
        for x in t[k]: yt[i,x]=1
        for x in p[k]: yp[i,x]=1
    res=dict(examples=len(t), labels=n, macro_F1=float(f1_score(yt,yp,average="macro",zero_division=0)),
             micro_F1=float(f1_score(yt,yp,average='micro',zero_division=0)), subset_acc=float(accuracy_score(yt,yp)))
    if verbose:
        print(f'  [official evaluator] Examples: {res["examples"]}  Labels: {res["labels"]}  Macro-F1: {res["macro_F1"]:.6f}  Micro-F1: {res["micro_F1"]:.6f}  Subset Accuracy: {res["subset_acc"]:.6f}')
    return res


def check_test_file(pred_path, test_ids, n_labels):
    """The checks Codabench applies to a submission (read_pred) + exactly one object per test case."""
    p = read_pred(pred_path, n_labels)
    if set(p) != set(test_ids) or len(p) != len(test_ids):
        raise ValueError(f"ID mismatch with test.json: missing={len(set(test_ids) - set(p))}, extra={len(set(p) - set(test_ids))}")
    return True


def dev_codabench_check(ids, Y, pred, out_dir, name, verbose=True):
    """Writes the dev truth (as {"id", "gold"}) and the predictions (as {"id", "pred"}) as files and scores them with the official evaluator code.
    Run `python evaluate_phase5.py --truth <truth> --prediction <prediction>` on the same files to get the same numbers."""
    os.makedirs(out_dir, exist_ok=True)
    assert all(isinstance(i, str) for i in ids), "case ids must be strings (the evaluator requires it)"
    Y, pred = np.asarray(Y).astype(bool), np.asarray(pred).astype(bool)
    tpath, ppath = os.path.join(out_dir, "dev_truth.json"), os.path.join(out_dir, f"{name}_dev_prediction.json")
    json.dump([dict(id=i, gold=[int(x) for x in np.flatnonzero(r)]) for i, r in zip(ids, Y)], open(tpath, "w"))
    json.dump([dict(id=i, pred=[int(x) for x in np.flatnonzero(r)]) for i, r in zip(ids, pred)], open(ppath, "w"))
    return official_eval(tpath, ppath, verbose), tpath, ppath

In [3]:
SWEEP_SUFFIX = ""
# every cached run of every Phase-5 notebook in one table (missing ones are skipped)
ALL = [('p5_gqa', 'gqa_lorma'), ('p5_gqa', 'gqa_partial4'), ('p5_moe', 'moe_partial2'), ('p5_gdn', 'gdn_lorma'), ('p5_lstm', 'lstm_full_ft'), ('p5_lstm', 'lstm_frozen'), ('p5_mor', 'mor_partial4'), ('p5_kd', 'kd_gqa_lorma'), ('p5_kd', 'kd_gqa_partial4'), ('p5_kd_lstm', 'kd_lstm_full_ft'), ('p5_moe_long', 'moe_partial2_long'), ('p5_moe_lorma', 'moe_lorma_long'), ('p5_gqa_long', 'gqa_partial4_long'), ('p5_gqa_long', 'gqa_lorma_long'), ('p5_gqa_long', 'gqa_lora_long'), ('p5_gdn2', 'gdn_partial4'), ('p5_gdn2', 'gdn_full_ft'), ('p5_mor2', 'mor_lorma'), ('p5_mor2', 'mor_full_ft'), ('p5_kd_long', 'kd_gqa_lorma_long'), ('p5_kd_long', 'kd_gqa_partial4_long'), ('p5_ablate_tech_moe', 't_partial'), ('p5_ablate_tech_moe', 't_lorma'), ('p5_ablate_tech_moe', 't_adalora'), ('p5_ablate_tech_moe', 't_adapters'), ('p5_ablate_tech_moe', 't_bitfit'), ('p5_ablate_tech_moe', 't_full_ft'), ('p5_ablate_tech_moe', 't_frozen'), ('p5_ablate_tech_moe', 't_qlora'), ('p5_ablate_design_moe', 'd_ref'), ('p5_ablate_design_moe', 'd_bce'), ('p5_ablate_design_moe', 'd_focal'), ('p5_ablate_design_moe', 'd_asl'), ('p5_ablate_design_moe', 'd_head_mean'), ('p5_ablate_design_moe', 'd_head_attn'), ('p5_ablate_design_moe', 'd_head_labelattn'), ('p5_ablate_design_moe', 'd_labelmix'), ('p5_ablate_design_moe', 'd_first4'), ('p5_ablate_design_moe', 'd_last4'), ('p5_ablate_design_moe', 'd_headtail4'), ('p5_ablate_design_moe', 'd_uniform4'), ('p5_ablate_design_moe', 'd_first1'), ('p5_ablate_design_moe', 'd_win256'), ('p5_ablate_design_moe', 'd_randominit'), ('p5_ablate_extra_moe', 'x_overlap'), ('p5_ablate_extra_moe', 'x_chunk_max'), ('p5_ablate_extra_moe', 'x_chunk_mean'), ('p5_ablate_extra_moe', 'x_cue4'), ('p5_ablate_extra_moe', 'x_win1024'), ('p5_ablate_extra_moe', 'x_pw_full'), ('p5_ablate_extra_moe', 'x_rare_sampling'), ('p5_ablate_extra_moe', 'x_prompt'), ('p5_ablate_extra_moe', 'x_prefix'), ('p5_ablate_extra_moe', 'x_all_chunks'), ('p5_lstm_long', 'kd_lstm_full_ft_long'), ('p5_mor_long', 'mor_full_ft_long'), ('p5_mor_long', 'kd_mor_full_ft_long'), ('p5_moe_attn', 'moe_attn_long'), ('p5_moe_attn_rare', 'moe_attn_rare_long'), ('p5_ablate_kd', 'k_none'), ('p5_ablate_kd', 'k_t2_l05'), ('p5_ablate_kd', 'k_t1_l05'), ('p5_ablate_kd', 'k_t4_l05'), ('p5_ablate_kd', 'k_t2_l025'), ('p5_ablate_kd', 'k_t2_l075'), ('p5_ablate_kd', 'k_t2_l1'), ('p5_ablate_lorma_moe', 'l_pi'), ('p5_ablate_lorma_moe', 'l_r8'), ('p5_ablate_lorma_moe', 'l_r32'), ('p5_ablate_lorma_moe', 'l_qv'), ('p5_kd_attn', 'kd_gqa_partial4_attn_long'), ('p5_kd_attn', 'kd_gqa_lorma_attn_long'), ('p5_ablate_more_moe', 'm_lora'), ('p5_ablate_more_moe', 'm_rslora'), ('p5_ablate_more_moe', 'm_ia3'), ('p5_ablate_more_moe', 'm_pfeiffer'), ('p5_ablate_more_moe', 'm_labelsmooth'), ('p5_ablate_tech', 't_lorma'), ('p5_ablate_tech', 't_adalora'), ('p5_ablate_tech', 't_adapters'), ('p5_ablate_tech', 't_bitfit'), ('p5_ablate_tech', 't_partial'), ('p5_ablate_tech', 't_full_ft'), ('p5_ablate_tech', 't_qlora'), ('p5_ablate_tech', 't_frozen'), ('p5_ablate_design', 'd_ref'), ('p5_ablate_design', 'd_bce'), ('p5_ablate_design', 'd_focal'), ('p5_ablate_design', 'd_asl'), ('p5_ablate_design', 'd_head_mean'), ('p5_ablate_design', 'd_head_attn'), ('p5_ablate_design', 'd_head_labelattn'), ('p5_ablate_design', 'd_labelmix'), ('p5_ablate_design', 'd_first4'), ('p5_ablate_design', 'd_last4'), ('p5_ablate_design', 'd_headtail4'), ('p5_ablate_design', 'd_uniform4'), ('p5_ablate_design', 'd_first1'), ('p5_ablate_design', 'd_win256'), ('p5_ablate_design', 'd_randominit')]
rows = []
for sweep, setup in ALL:
    try:
        r = torch.load(fetch_model_file(f"p5res_{sweep}{SWEEP_SUFFIX}__{setup}.pt"), map_location="cpu", weights_only=False)
    except Exception:
        continue
    s = r["scores"]; m = s["best_mode"]; t = s["tuned"][m]; g = s["tuned"]["global"]
    rows.append((sweep, setup, r["n_train_docs"], s["honest"]["global"], s["honest"][m], m, g["macro_F1"], t["micro_F1"], t["subset_acc"], t["macro_P"], t["macro_R"], r["best_epoch"],
                 r["info"]["params_trainable"] / 1e6, r["info"]["model_size_mb"], r["peak_mem_gb"] or 0, r["train_seconds"] / 60, r["infer_dev"]["docs_per_sec"], r["C"]["KD_TEACHER"] is not None))
rows.sort(key=lambda x: (x[2] < 10000, -x[4]))
print(f"{'notebook':18s} {'setup':20s} {'train n':>7s} {'honest-glob':>11s} {'honest-best':>11s} {'mode':>9s} {'F1 glob':>8s} {'microF1':>8s} {'subsetAcc':>9s} {'P':>6s} {'R':>6s} {'ep':>3s} {'train M':>8s} {'MB':>6s} {'GB':>5s} {'min':>6s} {'doc/s':>6s} KD")
for x in rows:
    print(f"{x[0]:18s} {x[1]:20s} {x[2]:7d} {x[3]:11.4f} {x[4]:11.4f} {x[5]:>9s} {x[6]:8.4f} {x[7]:8.4f} {x[8]:9.4f} {x[9]:6.3f} {x[10]:6.3f} {x[11]:3d} {x[12]:8.2f} {x[13]:6.0f} {x[14]:5.1f} {x[15]:6.1f} {x[16]:6.2f} {'yes' if x[17] else ''}")

notebook           setup                train n honest-glob honest-best      mode  F1 glob  microF1 subsetAcc      P      R  ep  train M     MB    GB    min  doc/s KD
p5_moe_attn        moe_attn_long          42750      0.3785      0.3842    shrunk   0.3799   0.4880    0.0528  0.385  0.408  14    43.50    754   1.9  254.9  32.16 
p5_moe_long        moe_partial2_long      42750      0.3732      0.3792    shrunk   0.3741   0.4870    0.0482  0.371  0.411  16    46.94    768   2.0  257.8  32.45 
p5_kd_attn         kd_gqa_partial4_attn_long   42750      0.3726      0.3777    shrunk   0.3740   0.4855    0.0453  0.377  0.407  16    25.16    324   1.3   89.4  63.64 yes
p5_kd_long         kd_gqa_partial4_long   42750      0.3691      0.3762    shrunk   0.3698   0.4826    0.0469  0.366  0.415  15    28.60    338   1.3   92.9  62.39 yes
p5_moe_lorma       moe_lorma_long         42750      0.3637      0.3721    shrunk   0.3654   0.4758    0.0406  0.356  0.418  14     4.29    770   2.4  438.0  28.5

In [4]:
import numpy as np, collections, itertools
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
SWEEP_SUFFIX = ""                       # (testing only)
MODELS = [('p5_moe_attn', 'moe_attn_long'), ('p5_moe_long', 'moe_partial2_long'), ('p5_kd_long', 'kd_gqa_partial4_long'), ('p5_kd_long', 'kd_gqa_lorma_long'), ('p5_gqa_long', 'gqa_lorma_long'), ('p5_gqa_long', 'gqa_partial4_long'), ('p5_kd', 'kd_gqa_lorma'), ('p5_moe', 'moe_partial2'), ('p5_gqa', 'gqa_lorma'), ('p5_gqa', 'gqa_partial4'), ('p5_kd', 'kd_gqa_partial4'), ('p5_gdn', 'gdn_lorma'), ('p5_lstm', 'lstm_full_ft')]                       # (notebook, setup): the runs to analyse (first = main model)
OUT = f"{WORK}/p5_analysis"; os.makedirs(OUT, exist_ok=True)
R = {}
for sweep, setup in MODELS:
    try:
        R[setup] = torch.load(fetch_model_file(f"p5res_{sweep}{SWEEP_SUFFIX}__{setup}.pt"), map_location="cpu", weights_only=False)
    except Exception as e:
        print(f"  (skipped {setup}: {type(e).__name__})")
names = [s for _, s in MODELS if s in R]
assert names, "no cached results found"
MAIN = names[0]
TRAIN_Y = None
try:
    P5 = "Phase_5_LSI"
    tr = json.load(open(get_dataset_file(f"{P5}/Train/train.json"), encoding="utf-8"))
    NL = R[MAIN]["dev_y"].shape[1]
    TRAIN_Y = np.stack([multi_hot(label_list(r["labels"]), NL) for r in tr]); del tr
except Exception as e:
    print("(train labels not available:", type(e).__name__, ")")
common = sorted(set.intersection(*[set(i for i, k in zip(R[n]["dev_ids"], R[n]["dev_sel_mask"]) if k) for n in names]))
pos = {n: {i: k for k, i in enumerate(R[n]["dev_ids"])} for n in names}
ix = {n: [pos[n][i] for i in common] for n in names}
Y = R[MAIN]["dev_y"][ix[MAIN]].astype(bool); NL = Y.shape[1]
P = {n: sigmoid_np(R[n]["dev_logits"][ix[n]]) for n in names}
MODE = {n: R[n]["scores"]["best_mode"] for n in names}
THR = {n: np.asarray(R[n]["scores"]["thr"][MODE[n]]) for n in names}
PRED = {n: apply_thr(P[n], THR[n]) for n in names}
NTOK = np.array([ (R[MAIN]["dev_attention"].get(i) or {}).get("n", 0) for i in common])
print(f"{len(common)} selection-set cases; main model {MAIN}; threshold mode per model: {MODE}")

# 1. overview
print("\n1. overview on the selection set (each model at its best threshold mode, fitted on the same cases = optimistic; 'honest' = cross-fit)")
print(f"{'model':22s} {'mode':>9s} {'macro-F1':>8s} {'honest':>7s} {'micro-F1':>8s} {'subsetAcc':>9s} {'macroP':>7s} {'macroR':>7s} {'pred/case':>9s} {'gold/case':>9s}")
for n in names:
    s = ml_scores(Y, PRED[n])
    print(f"{n:22s} {MODE[n]:>9s} {s['macro_F1']:8.4f} {R[n]['scores']['honest'][MODE[n]]:7.4f} {s['micro_F1']:8.4f} {s['subset_acc']:9.4f} {s['macro_P']:7.3f} {s['macro_R']:7.3f} {s['mean_pred_labels']:9.2f} {s['mean_true_labels']:9.2f}")

# 2. threshold strategies (global vs per-statute vs shrunk), precision/recall trade-off, min-one rule
print("\n2. thresholds, main model: macro-F1 by strategy (optimistic = fitted and scored on the same cases; honest = cross-fit), with / without 'at least one statute per case'")
sc = R[MAIN]["scores"]
for m in THR_MODES:
    t = sc["tuned"][m]
    t_no = ml_scores(Y, apply_thr(P[MAIN], sc["thr"][m], min_one=False))
    print(f"  {m:10s} optimistic {t['macro_F1']:.4f} | honest {sc['honest'][m]:.4f} (sd {sc['honest_sd'][m]:.4f}) | without min-one {t_no['macro_F1']:.4f} | micro {t['micro_F1']:.4f} | subset acc {t['subset_acc']:.4f}")
print(f"  at fixed 0.5: macro-F1 {sc['at05']['macro_F1']:.4f}, micro {sc['at05']['micro_F1']:.4f}, subset acc {sc['at05']['subset_acc']:.4f}")
print("  precision / recall trade-off with ONE global threshold (main model):")
print(f"  {'thr':>5s} {'macroP':>7s} {'macroR':>7s} {'macroF1':>8s} {'microP':>7s} {'microR':>7s} {'microF1':>8s} {'pred/case':>9s}")
for t in (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
    s = ml_scores(Y, apply_thr(P[MAIN], t))
    print(f"  {t:5.2f} {s['macro_P']:7.3f} {s['macro_R']:7.3f} {s['macro_F1']:8.4f} {s['micro_P']:7.3f} {s['micro_R']:7.3f} {s['micro_F1']:8.4f} {s['mean_pred_labels']:9.2f}")
tl = np.asarray(sc["thr"]["per_label"]); print(f"  per-statute thresholds: min {tl.min():.2f}, median {np.median(tl):.2f}, max {tl.max():.2f}; correlation with log training support: " + (f"{np.corrcoef(tl, np.log(TRAIN_Y.sum(0)))[0, 1]:.3f}" if TRAIN_Y is not None else "n/a"))

# 3. calibration (reliability), main model
print("\n3. calibration of the main model's probabilities (all statute x case entries): predicted-probability bin -> observed frequency")
pf, yf = P[MAIN].ravel(), Y.ravel().astype(float); bins = np.array([0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0001]); ece = 0.0
for a, b in zip(bins[:-1], bins[1:]):
    m = (pf >= a) & (pf < b)
    if m.any():
        ece += m.mean() * abs(pf[m].mean() - yf[m].mean()); print(f"  [{a:.2f}, {min(b, 1):.2f}) n={int(m.sum()):8d} mean predicted {pf[m].mean():.3f}  observed {yf[m].mean():.3f}")
print(f"  expected calibration error {ece:.4f}")

# 4. per statute: rare vs frequent, best / worst
f1 = np.array(ml_scores(Y, PRED[MAIN])["per_label_F1"]); sup_dev = Y.sum(0)
print("\n4. per statute (main model)")
if TRAIN_Y is not None:
    sup = TRAIN_Y.sum(0); order = np.argsort(sup); q = np.array_split(order, 4)
    print(f"  training support quartiles -> mean F1: " + ", ".join(f"Q{k + 1} (n={int(sup[g].min())}-{int(sup[g].max())}): {f1[g].mean():.3f}" for k, g in enumerate(q)) + f" | correlation F1 vs log support {np.corrcoef(f1, np.log(sup))[0, 1]:.3f}")
print("  10 hardest statutes (F1, training support, dev support): " + ", ".join(f"{l}: {f1[l]:.2f} ({int(TRAIN_Y[:, l].sum()) if TRAIN_Y is not None else '?'}, {int(sup_dev[l])})" for l in np.argsort(f1)[:10]))
print("  10 easiest statutes: " + ", ".join(f"{l}: {f1[l]:.2f} ({int(TRAIN_Y[:, l].sum()) if TRAIN_Y is not None else '?'}, {int(sup_dev[l])})" for l in np.argsort(-f1)[:10]))
plt.figure(figsize=(5, 3.5))
if TRAIN_Y is not None:
    plt.scatter(TRAIN_Y.sum(0), f1, s=10); plt.xscale("log"); plt.xlabel("training cases with the statute"); plt.ylabel("F1 on the selection set"); plt.tight_layout(); plt.savefig(f"{OUT}/f1_vs_support.png", dpi=150)
plt.close()

# 5. by number of gold statutes and by length
print("\n5. macro-F1 / exact-set accuracy by the number of gold statutes and by case length (main model)")
ng = Y.sum(1)
for lo, hi in ((1, 1), (2, 2), (3, 4), (5, 7), (8, 99)):
    m = (ng >= lo) & (ng <= hi)
    if m.sum() > 20:
        s = ml_scores(Y[m], PRED[MAIN][m]); print(f"  {lo}-{hi} gold statutes: {m.sum():5d} cases, samples-F1 {s['samples_F1']:.3f}, exact-set accuracy {s['subset_acc']:.3f}, predicted {s['mean_pred_labels']:.2f} vs gold {s['mean_true_labels']:.2f}")
if NTOK.any():
    q = np.quantile(NTOK, [0.25, 0.5, 0.75]); edges = [0, *q, NTOK.max() + 1]; b = np.digitize(NTOK, edges[1:-1])
    for k in range(4):
        m = b == k; s = ml_scores(Y[m], PRED[MAIN][m])
        print(f"  tokens {int(edges[k]):6d}-{int(edges[k + 1]):6d}: {m.sum():5d} cases, samples-F1 {s['samples_F1']:.3f}, exact-set accuracy {s['subset_acc']:.3f}, macro-F1 {s['macro_F1']:.3f}")
    cov = NTOK <= 4096
    for flag, nm in ((True, "fits in 8 x 512 tokens"), (False, "longer (middle not seen)")):
        m = cov == flag
        if m.sum() > 20: print(f"  {nm:26s} {m.sum():5d} cases: samples-F1 {ml_scores(Y[m], PRED[MAIN][m])['samples_F1']:.3f}")

# 6. label relations: chains that are predicted together, and the statutes that are confused / missed
print("\n6. label relationships (main model)")
G = Y.astype(float); co = G.T @ G; nn_ = np.diag(co); jac = co / (nn_[:, None] + nn_[None, :] - co)
np.fill_diagonal(jac, 0); pairs = sorted(((jac[a, b], a, b) for a in range(NL) for b in range(a + 1, NL)), reverse=True)[:8]
Pd = PRED[MAIN].astype(float); pco = Pd.T @ Pd
print("  most co-occurring gold pairs: gold Jaccard -> Jaccard of the predictions, and how often one is predicted correctly without the other")
for j, a, b in pairs:
    pj = pco[a, b] / max(Pd[:, a].sum() + Pd[:, b].sum() - pco[a, b], 1)
    both = (Y[:, a] & Y[:, b]); only_a = (PRED[MAIN][:, a] & Y[:, a] & ~PRED[MAIN][:, b] & Y[:, b]).sum()
    print(f"    {a:2d} & {b:2d}: gold {j:.2f} -> predicted {pj:.2f}; both gold in {int(both.sum())} cases, only {a} found while {b} missed in {int(only_a)}")
miss = (Y & ~PRED[MAIN]).astype(float); fpm = (~Y & PRED[MAIN]).astype(float)
conf = miss.T @ fpm                                   # conf[a, b]: cases where a is gold-but-missed and b is predicted-but-wrong
np.fill_diagonal(conf, 0)
top = sorted(((conf[a, b], a, b) for a in range(NL) for b in range(NL)), reverse=True)[:10]
print("  most frequent substitutions (statute missed -> wrong statute predicted in the same case): " + ", ".join(f"{a}->{b} ({int(c)})" for c, a, b in top))
fn_rate = miss.sum(0) / np.maximum(Y.sum(0), 1); fp_rate = fpm.sum(0) / np.maximum(PRED[MAIN].sum(0), 1)
print("  most often missed statutes (miss rate): " + ", ".join(f"{l}: {fn_rate[l]:.2f}" for l in np.argsort(-fn_rate)[:8]))
print("  statutes whose predictions are most often wrong (1 - precision): " + ", ".join(f"{l}: {fp_rate[l]:.2f}" for l in np.argsort(-fp_rate)[:8]))

# 7. where in the case the document head looks
ATT = R[MAIN].get("dev_attention") or {}
if ATT:
    bins10 = np.linspace(0, 1, 11); acc = np.zeros(10); cnt = np.zeros(10); first_last = collections.Counter()
    for i in common:
        a = ATT.get(i)
        if not a or not a["w"]: continue
        w = np.asarray(a["w"]); n_ = max(a["n"], 1)
        for wj, (s0, s1) in zip(w, a["spans"]):
            k = min(int(((s0 + s1) / 2) / n_ * 10), 9); acc[k] += wj * len(w); cnt[k] += 1
        first_last["last chunk has the largest weight" if int(np.argmax(w)) == len(w) - 1 else ("first chunk" if int(np.argmax(w)) == 0 else "a middle chunk")] += 1
    print("\n7. attention of the document head over chunk positions (weight x number of chunks; 1.0 = uniform), main model")
    print("position   " + " ".join(f"{int(bins10[b] * 100):>3d}-{int(bins10[b + 1] * 100):<3d}" for b in range(10)))
    print("attention  " + " ".join(f"{(acc[b] / cnt[b] if cnt[b] else float('nan')):7.2f}" for b in range(10)))
    print("chunk with the largest weight:", dict(first_last))
    plt.figure(figsize=(6, 3)); plt.plot((bins10[:-1] + bins10[1:]) / 2, [acc[b] / cnt[b] if cnt[b] else np.nan for b in range(10)], marker="o"); plt.axhline(1, color="grey", lw=0.5)
    plt.xlabel("chunk position in the case (0 = start, 1 = end)"); plt.ylabel("relative attention"); plt.tight_layout(); plt.savefig(f"{OUT}/attention_by_position.png", dpi=150); plt.close()

# 8. models: agreement and oracle
if len(names) > 1:
    print("\n8. pairwise disagreement of the models (share of statute x case decisions that differ) and macro-F1 of a simple vote")
    for a_, b_ in itertools.combinations(names, 2):
        print(f"  {a_:20s} vs {b_:20s} {(PRED[a_] != PRED[b_]).mean():.4f}")
    vote = np.mean([PRED[n] for n in names], 0) >= 0.5
    print(f"  majority vote of all {len(names)} models: macro-F1 {ml_scores(Y, vote)['macro_F1']:.4f} (best single {max(ml_scores(Y, PRED[n])['macro_F1'] for n in names):.4f})")
plt.figure(figsize=(6, 3.5))
for n in names[:6]:
    plt.plot(sorted(ml_scores(Y, PRED[n])["per_label_F1"]), label=n)
plt.xlabel("statutes (sorted by F1)"); plt.ylabel("F1"); plt.legend(fontsize=6); plt.tight_layout(); plt.savefig(f"{OUT}/per_statute_f1.png", dpi=150); plt.close()

# 9. per model: rare statutes and cases with many statutes (the brief asks how techniques differ there)
if TRAIN_Y is not None:
    sup = TRAIN_Y.sum(0); q = np.array_split(np.argsort(sup), 4); ng = Y.sum(1)
    print("\n9. per model: mean F1 of the rarest / most frequent quarter of statutes (by training support), samples-F1 on cases with 1 / 2-4 / 5+ gold statutes")
    print(f"{'model':22s} {'rare Q1 F1':>10s} {'Q2':>6s} {'Q3':>6s} {'frequent Q4':>11s} {'1 gold':>7s} {'2-4 gold':>8s} {'5+ gold':>8s}")
    for n in names:
        f = np.array(ml_scores(Y, PRED[n])["per_label_F1"])
        sf = [ml_scores(Y[m], PRED[n][m])["samples_F1"] for m in (ng == 1, (ng >= 2) & (ng <= 4), ng >= 5)]
        print(f"{n:22s} {f[q[0]].mean():10.3f} {f[q[1]].mean():6.3f} {f[q[2]].mean():6.3f} {f[q[3]].mean():11.3f} {sf[0]:7.3f} {sf[1]:8.3f} {sf[2]:8.3f}")

# 10. calibration techniques (main model): temperature scaling (one T) and per-statute Platt scaling (a_l * logit + b_l), fitted on half A, scored on half B
def _ece(p, y, bins=np.array([0, 0.02, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0001])):
    p, y = p.ravel(), y.ravel().astype(float); e = 0.0
    for a_, b_ in zip(bins[:-1], bins[1:]):
        m = (p >= a_) & (p < b_)
        if m.any(): e += m.mean() * abs(p[m].mean() - y[m].mean())
    return e
LG = R[MAIN]["dev_logits"][ix[MAIN]].astype(np.float64)
perm = np.random.RandomState(0).permutation(len(Y)); A, B = perm[:len(Y) // 2], perm[len(Y) // 2:]
def _bce(z, y): return float(np.mean(np.logaddexp(0, z) - y * z))
Ts = np.linspace(0.3, 3.0, 55); T = Ts[int(np.argmin([_bce(LG[A] / t, Y[A]) for t in Ts]))]
zA, yA = torch.tensor(LG[A]), torch.tensor(Y[A].astype(np.float64))
a_ = torch.ones(NL, dtype=torch.float64, requires_grad=True); b_ = torch.zeros(NL, dtype=torch.float64, requires_grad=True)
opt_ = torch.optim.LBFGS([a_, b_], max_iter=200, line_search_fn="strong_wolfe")
def _cl():
    opt_.zero_grad(); l = F.binary_cross_entropy_with_logits(zA * a_ + b_, yA) + 1e-3 * (((a_ - 1) ** 2).mean() + (b_ ** 2).mean()); l.backward(); return l       # mild pull to the identity
opt_.step(_cl)
cal = {"none": LG, f"temperature (T={T:.2f})": LG / T, "per-statute Platt": LG * a_.detach().numpy() + b_.detach().numpy()}
print("\n10. calibration (main model; fitted on half A of the selection set, scored on half B): ECE, log-loss, macro-F1 with thresholds fitted on A")
print(f"{'method':26s} {'ECE':>7s} {'log-loss':>8s} {'F1 global thr':>13s} {'F1 per-statute thr':>18s} {'F1 at 0.5':>9s}")
for k, z in cal.items():
    p = sigmoid_np(z)
    fg = ml_scores(Y[B], apply_thr(p[B], fit_thresholds(Y[A], p[A], "global")))["macro_F1"]
    fp = ml_scores(Y[B], apply_thr(p[B], fit_thresholds(Y[A], p[A], "per_label")))["macro_F1"]
    f5 = ml_scores(Y[B], apply_thr(p[B], 0.5))["macro_F1"]
    print(f"{k:26s} {_ece(p[B], Y[B]):7.4f} {_bce(z[B], Y[B]):8.4f} {fg:13.4f} {fp:18.4f} {f5:9.4f}")

# 11. post-processing (main model, its own thresholds): at-least-one rule, and a cap on the number of statutes per case
print("\n11. post-processing (main model, selection set): macro-F1 / micro-F1 / subset accuracy")
Pm, Tm = P[MAIN], THR[MAIN]
def _cap(pred, k):
    if k is None: return pred
    marg = np.where(pred, Pm - Tm, -np.inf); out = np.zeros_like(pred)
    top = np.argsort(-marg, 1)[:, :k]
    for j in range(k):
        sel = np.isfinite(marg[np.arange(len(marg)), top[:, j]]); out[np.arange(len(marg))[sel], top[sel, j]] = True
    return out
for nm, pred in (("no rule", apply_thr(Pm, Tm, min_one=False)), ("at least one statute", apply_thr(Pm, Tm)),
                 ("at least one, at most 8", _cap(apply_thr(Pm, Tm), 8)), ("at least one, at most 6", _cap(apply_thr(Pm, Tm), 6)), ("at least one, at most 4", _cap(apply_thr(Pm, Tm), 4))):
    s = ml_scores(Y, pred); print(f"  {nm:26s} {s['macro_F1']:.4f} / {s['micro_F1']:.4f} / {s['subset_acc']:.4f}  ({s['mean_pred_labels']:.2f} predicted per case)")
for f in os.listdir(OUT):
    upload_model_file(f"{OUT}/{f}", f"p5_analysis_{f}")
print(f"\nplots saved in {OUT} (and the bucket): {os.listdir(OUT)}")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

10079 selection-set cases; main model moe_attn_long; threshold mode per model: {'moe_attn_long': 'shrunk', 'moe_partial2_long': 'shrunk', 'kd_gqa_partial4_long': 'shrunk', 'kd_gqa_lorma_long': 'shrunk', 'gqa_lorma_long': 'shrunk', 'gqa_partial4_long': 'shrunk', 'kd_gqa_lorma': 'per_label', 'moe_partial2': 'per_label', 'gqa_lorma': 'shrunk', 'gqa_partial4': 'per_label', 'kd_gqa_partial4': 'shrunk', 'gdn_lorma': 'per_label', 'lstm_full_ft': 'per_label'}

1. overview on the selection set (each model at its best threshold mode, fitted on the same cases = optimistic; 'honest' = cross-fit)
model                       mode macro-F1  honest micro-F1 subsetAcc  macroP  macroR pred/case gold/case
moe_attn_long             shrunk   0.3906  0.3842   0.4880    0.0528   0.385   0.408      4.36      3.91
moe_partial2_long         shrunk   0.3831  0.3792   0.4870    0.0482   0.371   0.411      4.48      3.91
kd_gqa_partial4_long      shrunk   0.3816  0.3762   0.4826    0.0469   0.366   0.415      4.65

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

... (5 outputs left out)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Validating:           |   0%            

uploaded p5_analysis_per_statute_f1.png to bucket kavincg23/cs781-models

plots saved in /kaggle/working/p5_analysis (and the bucket): ['f1_vs_support.png', 'attention_by_position.png', 'per_statute_f1.png']


In [5]:
# ===================== Part C: statistical checks, KD diagnostics, learning curves, error examples =====================
COMPARE = [('KD vs no KD: GQA partial4, 16 ep', ('p5_kd_long', 'kd_gqa_partial4_long'), ('p5_gqa_long', 'gqa_partial4_long')), ('KD vs no KD: GQA LoRMA, 16 ep', ('p5_kd_long', 'kd_gqa_lorma_long'), ('p5_gqa_long', 'gqa_lorma_long')), ('KD vs no KD: MoR full FT, 16 ep', ('p5_mor_long', 'kd_mor_full_ft_long'), ('p5_mor_long', 'mor_full_ft_long')), ('KD vs no KD: LSTM full FT, 8 ep', ('p5_kd_lstm', 'kd_lstm_full_ft'), ('p5_lstm', 'lstm_full_ft')), ('KD vs no KD: GQA LoRMA, 8 ep (weaker teacher)', ('p5_kd', 'kd_gqa_lorma'), ('p5_gqa', 'gqa_lorma')), ('teacher MoE partial vs KD student GQA partial4', ('p5_moe_long', 'moe_partial2_long'), ('p5_kd_long', 'kd_gqa_partial4_long')), ('MoE: partial2 vs LoRMA, 16 ep', ('p5_moe_long', 'moe_partial2_long'), ('p5_moe_lorma', 'moe_lorma_long')), ('GQA: partial4 vs LoRA, 16 ep', ('p5_gqa_long', 'gqa_partial4_long'), ('p5_gqa_long', 'gqa_lora_long')), ('GQA: LoRA vs LoRMA, 16 ep', ('p5_gqa_long', 'gqa_lora_long'), ('p5_gqa_long', 'gqa_lorma_long')), ('MoE attention head vs BiLSTM head, 16 ep', ('p5_moe_attn', 'moe_attn_long'), ('p5_moe_long', 'moe_partial2_long')), ('KD students: attention head vs BiLSTM head (GQA partial4)', ('p5_kd_attn', 'kd_gqa_partial4_attn_long'), ('p5_kd_long', 'kd_gqa_partial4_long')), ('KD students: attention head vs BiLSTM head (GQA LoRMA)', ('p5_kd_attn', 'kd_gqa_lorma_attn_long'), ('p5_kd_long', 'kd_gqa_lorma_long')), ('MoE attention head + rare sampling vs attention head', ('p5_moe_attn_rare', 'moe_attn_rare_long'), ('p5_moe_attn', 'moe_attn_long')), ('longer training: MoE partial2 16 vs 8 ep', ('p5_moe_long', 'moe_partial2_long'), ('p5_moe', 'moe_partial2')), ('LSTM full FT vs frozen (Phase-1 representations)', ('p5_lstm', 'lstm_full_ft'), ('p5_lstm', 'lstm_frozen')), ('subset: attention head vs BiLSTM head (MoE)', ('p5_ablate_design_moe', 'd_head_attn'), ('p5_ablate_design_moe', 'd_ref')), ('subset: rare sampling vs uniform (MoE)', ('p5_ablate_extra_moe', 'x_rare_sampling'), ('p5_ablate_design_moe', 'd_ref')), ('subset: bce_pw vs plain BCE (MoE)', ('p5_ablate_design_moe', 'd_ref'), ('p5_ablate_design_moe', 'd_bce')), ('subset: LoRMA vs full FT (MoE)', ('p5_ablate_tech_moe', 't_lorma'), ('p5_ablate_tech_moe', 't_full_ft')), ('subset: frozen Phase-1 vs random init (MoE)', ('p5_ablate_tech_moe', 't_frozen'), ('p5_ablate_design_moe', 'd_randominit'))]          # (label, (notebook, setup) A, (notebook, setup) B): is A better than B? paired bootstrap on the same selection-set cases
KD_PAIRS = [(('p5_kd_long', 'kd_gqa_partial4_long'), ('p5_gqa_long', 'gqa_partial4_long'), ('p5_moe_long', 'moe_partial2_long')), (('p5_kd_long', 'kd_gqa_lorma_long'), ('p5_gqa_long', 'gqa_lorma_long'), ('p5_moe_long', 'moe_partial2_long')), (('p5_mor_long', 'kd_mor_full_ft_long'), ('p5_mor_long', 'mor_full_ft_long'), ('p5_moe_long', 'moe_partial2_long')), (('p5_kd_lstm', 'kd_lstm_full_ft'), ('p5_lstm', 'lstm_full_ft'), ('p5_moe', 'moe_partial2'))]         # (student with KD, same student without KD, teacher)
BOOT = 1000
_RC = {}
def _load(nb_setup):
    k = f"p5res_{nb_setup[0]}{SWEEP_SUFFIX}__{nb_setup[1]}.pt"
    if k not in _RC:
        try:
            _RC[k] = torch.load(fetch_model_file(k), map_location="cpu", weights_only=False)
        except Exception as e:
            _RC[k] = None; print(f"  (missing {k}: {type(e).__name__})")
    return _RC[k]
def _sel(r):
    """selection-set ids, gold, probabilities and predictions at the run's own best threshold mode"""
    m = np.asarray(r["dev_sel_mask"]); ids = [i for i, k in zip(r["dev_ids"], m) if k]
    P_ = sigmoid_np(r["dev_logits"])[m]; mode = r["scores"]["best_mode"]
    return ids, np.asarray(r["dev_y"])[m].astype(bool), P_, apply_thr(P_, np.asarray(r["scores"]["thr"][mode]))

def paired_bootstrap(Y_, pa, pb, B=BOOT, seed=0):
    """95% percentile interval of macro-F1(A) - macro-F1(B), resampling the SAME cases for both systems (Koehn 2004)."""
    rng = np.random.RandomState(seed); n = len(Y_); d = []
    for _ in range(B):
        ix_ = rng.randint(0, n, n)
        d.append(ml_scores(Y_[ix_], pa[ix_])["macro_F1"] - ml_scores(Y_[ix_], pb[ix_])["macro_F1"])
    d = np.array(d)
    return float(ml_scores(Y_, pa)["macro_F1"] - ml_scores(Y_, pb)["macro_F1"]), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d <= 0).mean())

print(f"\nC1. paired bootstrap ({BOOT} resamples of the selection-set cases; each run at its own best threshold mode, fitted on the same cases)")
print(f"{'comparison':58s} {'diff':>7s} {'95% CI':>18s} {'P(diff<=0)':>10s}")
BOOT_RES = {}
for lab, a, b in COMPARE:
    ra, rb = _load(a), _load(b)
    if ra is None or rb is None: continue
    ia, Ya, _, pa = _sel(ra); ib, Yb, _, pb = _sel(rb)
    common_ = sorted(set(ia) & set(ib)); pa_i = {i: k for k, i in enumerate(ia)}; pb_i = {i: k for k, i in enumerate(ib)}
    A_ = np.array([pa_i[i] for i in common_]); B_ = np.array([pb_i[i] for i in common_])
    d, lo, hi, p = paired_bootstrap(Ya[A_], pa[A_], pb[B_])
    BOOT_RES[lab] = dict(diff=d, lo=lo, hi=hi, p_le0=p)
    print(f"{lab:58s} {d:+7.4f} [{lo:+.4f}, {hi:+.4f}] {p:10.3f}" + ("   (CI excludes 0)" if lo > 0 or hi < 0 else ""))

print("\nC2. knowledge distillation: does the student follow the teacher, and does it copy the teacher's mistakes? (decisions = case x statute, selection set)")
print(f"{'student':26s} {'agree w/ teacher':>16s} {'exact-set agree':>15s} {'F1 where teacher right':>22s} {'teacher FP copied':>17s} {'teacher FN copied':>17s}")
for s_kd, s_ctl, t in KD_PAIRS:
    rs, rc, rt = _load(s_kd), _load(s_ctl), _load(t)
    if rs is None or rc is None or rt is None: continue
    it, Yt, _, pt = _sel(rt); ti = {i: k for k, i in enumerate(it)}
    for nm, r in ((s_kd[1] + " (KD)", rs), (s_ctl[1] + " (no KD)", rc)):
        i_, Y_, _, p_ = _sel(r); common_ = [i for i in i_ if i in ti]
        _pos = {i: k for k, i in enumerate(i_)}; S = np.array([_pos[i] for i in common_])
        T_ = np.array([ti[i] for i in common_]); Yc, ps, ptc = Y_[S], p_[S], pt[T_]
        agree = (ps == ptc).mean(); exact = (ps == ptc).all(1).mean()
        t_fp, t_fn = ptc & ~Yc, ~ptc & Yc
        right_cases = (ptc == Yc).all(1)
        f_right = ml_scores(Yc[right_cases], ps[right_cases])["samples_F1"] if right_cases.any() else float("nan")
        print(f"{nm:26s} {agree:16.4f} {exact:15.4f} {f_right:22.3f} {(ps & t_fp).sum() / max(t_fp.sum(), 1):17.3f} {((~ps) & t_fn).sum() / max(t_fn.sum(), 1):17.3f}")
print("  (teacher FP copied = share of the teacher's false positives the student also predicts; teacher FN copied = share of the statutes the teacher missed that the student also misses. "
      "Compare KD with no-KD: a higher copy rate under KD means the student also inherits the teacher's errors.)")

print("\nC3. learning curves (selection-sample macro-F1 at the best global threshold, after each epoch) -> learning_curves.png")
plt.figure(figsize=(7, 4))
_curves = {}
for nb_setup in [x for _, a, b in COMPARE for x in (a, b)] + [x for p in KD_PAIRS for x in p]:
    r = _load(nb_setup)
    if r is not None and nb_setup[1] not in _curves:
        _curves[nb_setup[1]] = [h["dev_macroF1_tuned"] for h in r["history"]]
for n, c in _curves.items():
    plt.plot(range(1, len(c) + 1), c, marker=".", label=n)
    print(f"  {n:26s} " + " ".join(f"{v:.3f}" for v in c))
plt.xlabel("epoch (6,000 training cases each)"); plt.ylabel("macro-F1 (selection sample)"); plt.legend(fontsize=6, ncol=2); plt.tight_layout()
plt.savefig(f"{OUT}/learning_curves.png", dpi=150); plt.close()

print("\nC4. example errors of the main model (first 3 selection-set cases with a missed AND a wrong statute, in file order)")
try:
    _dev = {r["id"]: r for r in json.load(open(get_dataset_file("Phase_5_LSI/Train/dev.json"), encoding="utf-8"))}
    shown = 0
    for k, i in enumerate(common):
        g, p = set(np.flatnonzero(Y[k])), set(np.flatnonzero(PRED[MAIN][k]))
        if (g - p) and (p - g):
            t = doc_text(_dev[i])
            print(f"--- case {i}: {len(t.split())} words | gold {sorted(g)} | predicted {sorted(p)} | missed {sorted(g - p)} | wrong {sorted(p - g)}\n    START: {t[:300]!r}\n    END:   {t[-300:]!r}")
            shown += 1
            if shown == 3: break
except Exception as e:
    print("  (dev texts not available:", type(e).__name__, ")")
json.dump(dict(bootstrap=BOOT_RES), open(f"{OUT}/analysis_extra.json", "w"), indent=1)


C1. paired bootstrap (1000 resamples of the selection-set cases; each run at its own best threshold mode, fitted on the same cases)
comparison                                                    diff             95% CI P(diff<=0)
KD vs no KD: GQA partial4, 16 ep                           +0.0108 [+0.0066, +0.0154]      0.000   (CI excludes 0)
KD vs no KD: GQA LoRMA, 16 ep                              +0.0091 [+0.0050, +0.0132]      0.000   (CI excludes 0)
KD vs no KD: MoR full FT, 16 ep                            +0.0285 [+0.0243, +0.0325]      0.000   (CI excludes 0)
KD vs no KD: LSTM full FT, 8 ep                            +0.0243 [+0.0209, +0.0279]      0.000   (CI excludes 0)
KD vs no KD: GQA LoRMA, 8 ep (weaker teacher)              +0.0156 [+0.0117, +0.0195]      0.000   (CI excludes 0)
teacher MoE partial vs KD student GQA partial4             +0.0014 [-0.0029, +0.0058]      0.261
MoE: partial2 vs LoRMA, 16 ep                              +0.0061 [+0.0019, +0.0106]      0.004  

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

--- case 100022754: 55 words | gold [np.int64(5), np.int64(55), np.int64(58)] | predicted [np.int64(5), np.int64(43), np.int64(45), np.int64(46), np.int64(51), np.int64(61), np.int64(83), np.int64(97)] | missed [np.int64(55), np.int64(58)] | wrong [np.int64(43), np.int64(45), np.int64(46), np.int64(51), np.int64(61), np.int64(83), np.int64(97)]
    START: 'In re: Application <ENTITY> anticipatory bail under <SECTION> of the <ACT> filed on 09.12.2013 in connection with Dhubulia Police Station Case No. 71/2013 dated 15.03.2013 <ENTITY> commission of offence punishable under <SECTION> of the <ACT>. And In the matter of: <ENTITY> & Ors. This application <'
    END:   ' filed on 09.12.2013 in connection with Dhubulia Police Station Case No. 71/2013 dated 15.03.2013 <ENTITY> commission of offence punishable under <SECTION> of the <ACT>. And In the matter of: <ENTITY> & Ors. This application <ENTITY> anticipatory bail is, thus, allowed. (<ENTITY>, J.) (<ENTITY>, J.)'
--- case 100031854: 625 w